# Session 1: two shops are steadier than one
**A. Real-world problem.** You have some savings to put into a small business. One shop
sells umbrellas, another sells ice cream. On rainy days the umbrella shop earns a lot and the
ice cream shop earns little; on dry days it is the other way round. If you put all your money
in one shop, your income jumps up and down with the weather. If you split it, the ups of one
shop cover the downs of the other.

**Safety note.** This is a toy model for learning. It is NOT investment advice. The shops
are made up, the income numbers are made up, and real markets do not behave like a coin flip
of weather. Past results never predict the future.

**Plan.** (1) write each shop's income on a rainy and a dry day, (2) simulate 365 days with a
seeded random generator, (3) measure the average and the swing of the income, (4) split the
money between the shops and watch the swing shrink.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# income per day, in made-up money units, when all the money is in that shop
UMBRELLA = {"rain": 100.0, "dry": 20.0}
ICE_CREAM = {"rain": 30.0, "dry": 90.0}
P_RAIN = 0.5          # chance that a day is rainy

## Step 1: the weather and the shop incomes
`simulate_weather` gives a list of True (rainy) / False (dry) days from a seeded generator,
so everyone gets the same days. `shop_income` turns the weather into income.

In [ ]:
def simulate_weather(n_days, seed=1, p_rain=P_RAIN):
    """True for a rainy day, False for a dry day (seeded, repeatable)."""
    rng = np.random.default_rng(seed)
    return rng.random(n_days) < p_rain


def shop_income(shop, rain):
    """Income for each day: shop['rain'] on rainy days, shop['dry'] on dry days."""
    return np.where(rain, shop["rain"], shop["dry"])

## Step 2: split the money
Put a fraction `f` of the money into the umbrella shop and `1 - f` into the ice cream shop.
Your income each day is the same split of the two shop incomes:

    income = f * umbrella + (1 - f) * ice cream

`f = 1` is "all eggs in the umbrella basket"; `f = 0.5` is half and half.

In [ ]:
def split_income(rain, f):
    return f * shop_income(UMBRELLA, rain) + (1 - f) * shop_income(ICE_CREAM, rain)

## Step 3: how steady is the income?
* **Average** = the sum of the daily incomes divided by the number of days.
* **Swing** (standard deviation) = the typical distance of a day from the average. A small
  swing means a steady income. NumPy's `std` computes it.

With only two kinds of day we can also compute both exactly. With rain chance p, the income
is `R` on rainy days and `D` on dry days, so
average = p R + (1 - p) D and swing = |R - D| * sqrt(p (1 - p)).

In [ ]:
def exact_mean_std(f, p_rain=P_RAIN):
    rainy = float(split_income(np.array([True]), f)[0])
    dry = float(split_income(np.array([False]), f)[0])
    mean = p_rain * rainy + (1 - p_rain) * dry
    std = abs(rainy - dry) * np.sqrt(p_rain * (1 - p_rain))
    return mean, std


def sample_mean_std(f, n_days=365, seed=1):
    inc = split_income(simulate_weather(n_days, seed), f)
    return float(inc.mean()), float(inc.std())


def zero_swing_split():
    """The split f that makes rainy-day income equal dry-day income (swing = 0).
    Solve f*100 + (1-f)*30 = f*20 + (1-f)*90."""
    a = UMBRELLA["rain"] - ICE_CREAM["rain"] - (UMBRELLA["dry"] - ICE_CREAM["dry"])
    b = ICE_CREAM["dry"] - ICE_CREAM["rain"]
    return b / a

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    rain = simulate_weather(365, seed=1)
    print(f"rainy days in 365: {int(rain.sum())}")
    for name, f in [("all umbrella", 1.0), ("all ice cream", 0.0), ("half and half", 0.5)]:
        m, s = sample_mean_std(f)
        em, es = exact_mean_std(f)
        print(f"{name:14s} sample: mean {m:6.2f} swing {s:5.2f}   exact: mean {em:6.2f} swing {es:5.2f}")
    fz = zero_swing_split()
    print(f"split with zero swing: f = {fz:.4f} (mean {exact_mean_std(fz)[0]:.1f})")

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
    days = np.arange(1, 31)
    for ax, (name, f) in zip(axes, [("all umbrella", 1.0), ("all ice cream", 0.0), ("half and half", 0.5)]):
        inc = split_income(rain, f)[:30]
        ax.bar(days, inc, color=["#2a6fdb" if r else "#e0a030" for r in rain[:30]])
        ax.set_title(f"{name}: swing {sample_mean_std(f)[1]:.1f}")
        ax.set_xlabel("day (blue = rain, orange = dry)")
    axes[0].set_ylabel("income per day (money units)")
    fig.tight_layout()
    fig.savefig(OUT / "shop_incomes.png", dpi=120)

    fs = np.linspace(0, 1, 101)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(100 * fs, [exact_mean_std(f)[1] for f in fs], label="exact swing")
    ax2.plot(100 * fs[::10], [sample_mean_std(f)[1] for f in fs[::10]], "o", label="365 simulated days")
    ax2.axvline(100 * fz, color="gray", ls=":")
    ax2.set_xlabel("money in the umbrella shop (%)")
    ax2.set_ylabel("swing of daily income (money units)")
    ax2.set_title("The swing is smallest near a 43% umbrella split")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "swing_vs_split.png", dpi=120)

## Where this stops
Real businesses (and real investments) are not perfectly opposite, and their numbers are not
known in advance. Session 2 measures "how opposite" with a number called correlation.

## Coding tasks
1. **Easy.** Change the shop incomes in `lesson.py` to umbrella 90 / 10 and ice cream 20 / 80. Predict the zero-swing split before you run it, then check with `zero_swing_split`.
2. **Stretch.** Write `swing_table(splits)` that prints the exact swing for splits 0, 10, ..., 100 percent and check that the smallest entry is near 43 percent.
3. **Explain.** Explain in two sentences why two umbrella shops would not make your income steadier.

## Assessment
- **A. Conceptual.** Why does putting money in both shops make the income steadier, even though each shop is unsteady?
- **B. Numerical.** Put 25 percent of the money in the umbrella shop. Find the rainy-day income, the dry-day income, the average and the swing (rain chance 0.5).
- **C. Graph and simulation interpretation.** In the graph of swing against split, where is the swing zero and what are the swings at 0 and at 100 percent? What does the V shape mean on each side of the bottom?
- **D. Design and optimisation challenge.** Design: you want the swing to be at most 10 money units per day. Which splits work? Why might you not choose exactly the zero-swing split?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    rain = L.simulate_weather(365, seed=1)
    checks.append(("the weather is repeatable for a given seed", np.array_equal(rain, L.simulate_weather(365, seed=1))))
    checks.append(("the weather differs for a different seed", not np.array_equal(rain, L.simulate_weather(365, seed=2))))
    checks.append(("shop_income gives 100 on rain and 20 on dry for umbrellas",
                   np.allclose(L.shop_income(L.UMBRELLA, np.array([True, False])), [100.0, 20.0])))
    # hand calculation: all umbrella, p = 0.5: mean 60, swing 40
    m, s = L.exact_mean_std(1.0)
    checks.append(("hand calculation: all umbrella has average 60 and swing 40", abs(m - 60) < 1e-9 and abs(s - 40) < 1e-9))
    m, s = L.exact_mean_std(0.0)
    checks.append(("hand calculation: all ice cream has average 60 and swing 30", abs(m - 60) < 1e-9 and abs(s - 30) < 1e-9))
    m, s = L.exact_mean_std(0.5)
    checks.append(("worked example: half and half has average 60 and swing 5 (income 65 rain, 55 dry)",
                   abs(m - 60) < 1e-9 and abs(s - 5) < 1e-9
                   and np.allclose(L.split_income(np.array([True, False]), 0.5), [65.0, 55.0])))
    fz = L.zero_swing_split()
    checks.append(("the zero-swing split is 3/7 = 0.4286 and gives swing 0",
                   abs(fz - 3 / 7) < 1e-12 and L.exact_mean_std(fz)[1] < 1e-9))
    checks.append(("the average income is 60 for every split (the made-up numbers are chosen so)",
                   all(abs(L.exact_mean_std(f)[0] - 60) < 1e-9 for f in np.linspace(0, 1, 11))))
    # simulation agrees with the exact numbers
    sm, ss = L.sample_mean_std(0.5, n_days=20000, seed=3)
    checks.append(("20000 simulated days: half-and-half average within 0.5 of 60 and swing within 0.5 of 5",
                   abs(sm - 60) < 0.5 and abs(ss - 5) < 0.5))
    s1 = L.sample_mean_std(1.0)[1]
    s5 = L.sample_mean_std(0.5)[1]
    checks.append(("365 simulated days: splitting cuts the swing from about 40 to about 5 (all umbrella > 35, half < 7)",
                   s1 > 35 and s5 < 7))
    checks.append(("seed 1: 189 rainy days; sample averages 61.4 (umbrella), 58.9 (ice cream), 60.2 (half); swing 40.0 / 30.0 / 5.0",
                   int(rain.sum()) == 189
                   and abs(L.sample_mean_std(1.0)[0] - 61.42) < 0.01 and abs(L.sample_mean_std(0.0)[0] - 58.93) < 0.01
                   and abs(L.sample_mean_std(0.5)[0] - 60.18) < 0.01
                   and abs(L.sample_mean_std(1.0)[1] - 39.97) < 0.01 and abs(L.sample_mean_std(0.0)[1] - 29.98) < 0.01
                   and abs(L.sample_mean_std(0.5)[1] - 5.00) < 0.01))
    r25 = L.split_income(np.array([True, False]), 0.25)
    checks.append(("25% umbrella: income 47.5 on rain and 72.5 on dry, average 60 and swing 12.5",
                   np.allclose(r25, [47.5, 72.5]) and abs(L.exact_mean_std(0.25)[0] - 60) < 1e-9 and abs(L.exact_mean_std(0.25)[1] - 12.5) < 1e-9))
    checks.append(("design: swing is at most 10 exactly for splits between 2/7 = 28.6% and 4/7 = 57.1%",
                   abs(L.exact_mean_std(2 / 7)[1] - 10) < 1e-9 and abs(L.exact_mean_std(4 / 7)[1] - 10) < 1e-9
                   and L.exact_mean_std(0.25)[1] > 10 and L.exact_mean_std(0.6)[1] > 10 and L.exact_mean_std(0.5)[1] < 10))
    # edge: swing is 0 when it always rains
    checks.append(("edge: if it always rains the swing is 0", L.exact_mean_std(1.0, p_rain=1.0)[1] == 0.0))
    ok = all(L.exact_mean_std(f)[1] < 30 for f in np.linspace(0.15, 0.7, 12))
    checks.append(("every split from 15% to 70% umbrella is steadier than either shop alone (swing < 30)", ok))
    # swing of the formula |70 f - 30| used in the app, at p = 0.5
    checks.append(("swing equals |70 f - 30| at rain chance 0.5",
                   all(abs(L.exact_mean_std(f)[1] - abs(70 * f - 30)) < 1e-9 for f in np.linspace(0, 1, 11))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
